# 04 - Análisis exploratorio de datos (EDA)

Antes de entrenar cualquier modelo, miramos los datos de `demanda_gold` para entenderlos. Queremos responder seis preguntas:

| Sección | Pregunta |
|---|---|
| 1. Calidad | ¿Faltan datos? ¿Dónde y por qué? |
| 2. Lo que queremos predecir | ¿Cómo se reparten los valores de `pasajeros`? ¿Conviene transformarlo? |
| 3. Patrones en el tiempo | ¿Cómo cambia la cantidad de pasajeros según la hora y el tipo de día? |
| 4. Líneas | ¿Qué tan distintas son las líneas entre sí? |
| 5. Calendario y clima | ¿Las clases, los feriados y la lluvia cambian la cantidad de pasajeros? |
| 6. Relación entre variables | ¿Qué variables se parecen a `pasajeros`? ¿Cuáles dicen lo mismo entre sí? |

**Regla importante: el EDA se hace solo con `train`.** Si miramos val o test para decidir cosas (qué variables usar, qué transformación hacer), estamos haciendo trampa sin querer. Val y test se usan recién en los notebooks 05 y 06.

In [0]:
%run ./00_setup

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from bondis import variables

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})

# El tipo de día (hábil, sábado, domingo, turismo, feriado) se calcula en src/bondis/variables.py.
# Usamos la misma función en todos los notebooks para que siempre signifique lo mismo.
TIPOS_DIA = variables.TIPOS_DIA
con_tipo_dia = variables.con_tipo_dia
COLORES = {"hábil": "tab:blue", "sábado": "tab:orange", "domingo": "tab:green",
           "turismo": "tab:purple", "feriado": "tab:red"}


gold = spark.table(config.TABLE_GOLD)
train = con_tipo_dia(gold.where(F.col("particion") == "train")).cache()

# Pasajeros de todo el sistema por día. Lo usan varias secciones de abajo.
por_dia = (
    train.groupBy("fecha", "tipo_dia", "dia_semana", "es_feriado", "es_eleccion",
                  "clases_primaria", "clases_media", "nombre_dia_especial")
    .agg(F.sum("pasajeros").alias("pasajeros"))
    .toPandas()
    .sort_values("fecha")
)
por_dia["fecha"] = pd.to_datetime(por_dia["fecha"])
print(f"Días en train: {len(por_dia)}")

## 1. Calidad de los datos

Lo que esperamos encontrar:
- **Muchos ceros**: por las horas de la madrugada y por líneas que no andan todos los días. Es normal.
- **Rezagos vacíos** en las primeras 4 semanas de 2024: no hay días anteriores para mirar.
- **Clima vacío** antes del 19/01/2024: ahí empiezan los pronósticos guardados.
- **`nombre_dia_especial` vacío** casi siempre: solo tiene algo en días especiales (feriados, elecciones).

Si aparece algún otro vacío, es una sorpresa y hay que investigarlo antes de seguir.

In [0]:
r = train.agg(
    F.count("*").alias("filas"),
    F.countDistinct("linea").alias("lineas"),
    F.min("fecha").alias("desde"),
    F.max("fecha").alias("hasta"),
    F.sum("pasajeros").alias("pasajeros"),
    F.avg((F.col("pasajeros") == 0).cast("int")).alias("prop_ceros"),
    F.avg(F.col("fila_completa").cast("int")).alias("prop_completas"),
    F.min(F.when(F.col("fila_completa"), F.col("fecha"))).alias("primera_completa"),
).first()

print(f"Filas:                {r['filas']:,}")
print(f"Líneas:               {r['lineas']}")
print(f"Fechas:               {r['desde']} a {r['hasta']}")
print(f"Pasajeros:            {r['pasajeros']:,}")
print(f"Horas con 0 pasajeros: {r['prop_ceros']:.1%}")
print(f"Filas completas:      {r['prop_completas']:.1%} (desde {r['primera_completa']})")

nulos = train.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in train.columns]).first().asDict()
nulos = pd.Series(nulos)
nulos = (nulos[nulos > 0] / r["filas"]).sort_values(ascending=False).rename("proporción vacía")
print("\nColumnas con vacíos:")
print(nulos.map("{:.2%}".format).to_string())

## 2. Lo que queremos predecir: `pasajeros`

Calculamos algunos números para describir cómo se reparten los valores:

- **Media**: el promedio.
- **Mediana**: el valor del medio si ordenamos todo de menor a mayor. La mitad de las horas tiene menos y la otra mitad tiene más.
- **Desvío**: qué tan lejos de la media están los valores, en promedio.
- **Asimetría**: dice si los datos están "torcidos" para un lado. Si da cerca de 0, el histograma es parejo a los dos lados. Si da positiva y grande, hay una **cola larga a la derecha**: muchas horas con pocos pasajeros y pocas horas con muchísimos.
- **Curtosis**: dice qué tanto pesan los valores muy extremos. Si da alta, hay valores muy grandes que se alejan mucho del resto.

**¿Por qué nos importa?** Si la media es mucho más grande que la mediana, hay pocas horas pico con muchísima gente. Esas horas dominan el error del modelo. Una solución es usar $\log(1 + \text{pasajeros})$ en lugar de `pasajeros`. El logaritmo achica los números grandes mucho más que los chicos (por ejemplo, 10 pasa a 2,4 y 1000 pasa a 6,9), y así el histograma queda más parejo. Usamos 1 + pasajeros porque el logaritmo de 0 no existe. Si lo usamos, al final deshacemos la transformación para que el error se mida siempre en pasajeros.

In [0]:
d = train.agg(
    F.mean("pasajeros").alias("media"),
    F.stddev("pasajeros").alias("desvio"),
    F.skewness("pasajeros").alias("asimetria"),
    F.kurtosis("pasajeros").alias("curtosis"),
    F.max("pasajeros").alias("maximo"),
    F.expr("percentile_approx(pasajeros, array(0.25, 0.5, 0.75, 0.9, 0.99))").alias("q"),
).first()
q25, q50, q75, q90, q99 = d["q"]

print(f"Media:     {d['media']:8.1f}    Mediana: {q50}")
print(f"Desvío:    {d['desvio']:8.1f}    Q1 / Q3: {q25} / {q75}")
print(f"Asimetría: {d['asimetria']:8.2f}    P90 / P99: {q90} / {q99}")
print(f"Curtosis:  {d['curtosis']:8.2f}    Máximo: {d['maximo']}")

In [0]:
# Los histogramas se calculan en Spark y solo se traen los conteos, así es rápido
ancho = max(1, int(np.ceil(q99 / 50)))
hist_lin = (train.where(F.col("pasajeros") <= q99)
            .groupBy((F.floor(F.col("pasajeros") / ancho) * ancho).alias("x")).count()
            .toPandas().sort_values("x"))
hist_log = (train.groupBy((F.floor(F.log1p("pasajeros") * 5) / 5).alias("x")).count()
            .toPandas().sort_values("x"))

fig, (a1, a2) = plt.subplots(1, 2)
a1.bar(hist_lin["x"], hist_lin["count"], width=ancho, align="edge")
a1.set(title=f"Pasajeros por hora (hasta el P99 = {q99})", xlabel="pasajeros", ylabel="filas")
a2.bar(hist_log["x"], hist_log["count"], width=0.2, align="edge")
a2.set(title="log(1 + pasajeros)", xlabel="log(1 + pasajeros)", ylabel="filas")
plt.tight_layout()
plt.show()

## 3. Patrones en el tiempo

### 3.1 Pasajeros por hora según el tipo de día

Para cada tipo de día, el promedio de pasajeros de todo el sistema en cada hora. Qué mirar:
- ¿Los días hábiles tienen dos picos (mañana y tarde)? ¿A qué hora?
- ¿Los sábados y domingos tienen otra **forma**, o solo tienen menos gente?

**¿Por qué importa?** Si las curvas tienen formas distintas, no alcanza con un modelo que diga "la hora suma tanto y el tipo de día suma tanto". Ejemplo: a las 7 de la mañana un día hábil está lleno, pero un domingo está vacío. El efecto de la hora **depende** del tipo de día. A eso se le llama **interacción** entre hora y tipo de día. En el modelo lineal hay que agregarla a mano.

In [0]:
perfil = (
    train.groupBy("fecha", "tipo_dia", "hora").agg(F.sum("pasajeros").alias("p"))
    .groupBy("tipo_dia", "hora").agg(F.avg("p").alias("pasajeros"), F.countDistinct("fecha").alias("dias"))
    .toPandas()
)

fig, ax = plt.subplots()
for t in TIPOS_DIA:
    s = perfil[perfil["tipo_dia"] == t].sort_values("hora")
    if len(s):
        ax.plot(s["hora"], s["pasajeros"], marker="o", ms=3, color=COLORES[t], label=f"{t} ({s['dias'].iloc[0]} días)")
ax.set(title="Perfil horario del sistema", xlabel="hora", ylabel="pasajeros promedio (todo el sistema)",
       xticks=range(0, 24, 2))
ax.legend()
plt.show()

### 3.2 Pasajeros de cada día

Total de todo el sistema, día por día, en train. Qué mirar:
- El ciclo de la semana (bajan los fines de semana).
- Los días muy bajos: ¿todos tienen explicación en el calendario (feriados, Turismo, días sin ómnibus)? Si hay un día bajo **sin explicación**, puede ser un paro. Hay que anotarlo.
- Cambios de nivel: el comienzo de las clases en marzo, las vacaciones de julio.

In [0]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(por_dia["fecha"], por_dia["pasajeros"], color="lightgray", lw=1, zorder=1)
for t in TIPOS_DIA:
    s = por_dia[por_dia["tipo_dia"] == t]
    ax.scatter(s["fecha"], s["pasajeros"], s=10, color=COLORES[t], label=t, zorder=2)
ax.set(title="Pasajeros por día (todo el sistema)", ylabel="pasajeros")
ax.legend(ncol=5)
plt.show()

# Días hábiles muy bajos (menos del 70 % de lo normal en su mes). Podrían ser paros.
habiles = por_dia[por_dia["tipo_dia"] == "hábil"].copy()
habiles["mediana_mes"] = habiles.groupby(habiles["fecha"].dt.month)["pasajeros"].transform("median")
habiles["ratio"] = habiles["pasajeros"] / habiles["mediana_mes"]
bajos = habiles[habiles["ratio"] < 0.7][["fecha", "pasajeros", "ratio", "nombre_dia_especial"]]
print("Días hábiles con menos del 70 % de la mediana de su mes:")
print(bajos.round({"ratio": 3}).to_string(index=False) if len(bajos) else "  ninguno")

## 4. Diferencias entre líneas

Qué mirar:
- **Concentración**: ¿cuántos pasajeros llevan las 10 o 20 líneas más grandes? Si es la mayoría, el error total del modelo lo van a decidir esas pocas líneas.
- **Media contra desvío**: si las líneas con más pasajeros también varían más (los puntos forman una recta en el gráfico), el error de cada línea crece con su tamaño. Es otra razón para usar $\log(1 + \text{pasajeros})$.

In [0]:
lineas = (
    train.groupBy("linea", "tipo_linea")
    .agg(F.sum("pasajeros").alias("total"),
         F.avg("pasajeros").alias("media_hora"),
         F.stddev("pasajeros").alias("desvio_hora"),
         F.avg((F.col("pasajeros") == 0).cast("int")).alias("prop_ceros"))
    .toPandas().sort_values("total", ascending=False).reset_index(drop=True)
)
lineas["acumulado"] = lineas["total"].cumsum() / lineas["total"].sum()
for k in (10, 20, 50):
    if k <= len(lineas):
        print(f"Las {k} líneas más grandes llevan el {lineas['acumulado'].iloc[k - 1]:.1%} de los pasajeros")

fig, (a1, a2) = plt.subplots(1, 2)
a1.plot(np.arange(1, len(lineas) + 1), lineas["acumulado"], marker=".")
a1.set(title="Concentración de pasajeros", xlabel="cantidad de líneas (de mayor a menor)",
       ylabel="proporción acumulada", ylim=(0, 1.02))
a2.scatter(lineas["media_hora"], lineas["desvio_hora"], s=12)
a2.set(title="Media contra desvío por línea", xlabel="pasajeros por hora (media)",
       ylabel="desvío", xscale="log", yscale="log")
plt.tight_layout()
plt.show()

print("Las 5 líneas más chicas:")
print(lineas.tail(5)[["linea", "total", "media_hora", "prop_ceros"]].to_string(index=False))

## 5. Calendario y clima

### 5.1 Clases y feriados

Promedio de pasajeros por día según el tipo de día y si hay clases. Qué mirar:
- En días hábiles, ¿cuánta más gente viaja cuando hay clases?
- Los feriados laborables (el comercio abre), ¿se parecen más a un día hábil o a un domingo?
- Los domingos de elecciones, ¿son distintos a un domingo común?

In [0]:
def tabla(df, por):
    return (df.groupby(por)["pasajeros"].agg(dias="count", promedio="mean")
            .round(0).astype({"promedio": "int64"}))

h = por_dia[por_dia["tipo_dia"] == "hábil"]
print("Días hábiles, según clases:")
print(tabla(h, ["clases_primaria", "clases_media"]).to_string(), "\n")

print("Días hábiles, feriado laborable o no:")
print(tabla(h, ["es_feriado"]).to_string(), "\n")

print("Domingos, con elecciones o sin:")
print(tabla(por_dia[por_dia["dia_semana"] == 7], ["es_eleccion"]).to_string())

### 5.2 Lluvia y temperatura pronosticadas

**El problema.** Si comparamos directo los días con lluvia y sin lluvia, se mezclan otras cosas. Por ejemplo, en invierno llueve más, y en invierno también hay clases. Entonces no sabríamos si la diferencia es por la lluvia o por las clases.

**La solución.** Comparamos cada día con lo que es "normal" para ese día:

$$\text{cociente del día} = \frac{\text{pasajeros del día}}{\text{suma de las medianas de 4 semanas del día}}$$

La mediana de las 4 semanas anteriores (misma línea, misma hora, mismo día de la semana) ya tiene en cuenta la época del año y el día de la semana. Si el cociente da 0,95, ese día viajó un 5 % menos de gente que lo normal. Si da 1,10, un 10 % más. Se usan solo días hábiles con filas completas.

**Cuidado con los cambios de época.** La mediana mira 4 semanas para atrás. El primer mes de clases se compara con semanas de vacaciones, así que el cociente sale alto aunque no haya pasado nada con el clima. Como marzo es caluroso, parecería que el calor hace viajar más gente, y es mentira. Por eso se sacan los días en que el estado de las clases (primaria y media) no es igual al de las 4 semanas con las que se comparan.

Mirá siempre la columna `dias`. Si hay pocos días en un grupo, una diferencia de 2 o 3 puntos puede ser casualidad.

In [0]:
clima_dia = (
    con_tipo_dia(gold.where((F.col("particion") == "train") & F.col("fila_completa")))
    .where(F.col("tipo_dia") == "hábil")
    .groupBy("fecha")
    .agg((F.sum("pasajeros") / F.sum("pasajeros_mediana_4sem")).alias("cociente"),
         F.first("lluvia_pron_dia_mm").alias("lluvia_dia"),
         F.avg("temperatura_pron").alias("temp_media"))
    .toPandas()
)
clima_dia["fecha"] = pd.to_datetime(clima_dia["fecha"])

# Solo dejamos los días que tienen el mismo estado de clases que 7, 14, 21 y 28 días antes
estado = por_dia.set_index("fecha")[["clases_primaria", "clases_media"]]


def regimen_estable(f):
    refs = [f - pd.Timedelta(days=k) for k in (7, 14, 21, 28)]
    return all(r in estado.index and (estado.loc[r] == estado.loc[f]).all() for r in refs)


antes = len(clima_dia)
clima_dia = clima_dia[clima_dia["fecha"].map(regimen_estable)]
print(f"Días hábiles completos: {antes}. Con el mismo estado de clases que en las 4 semanas anteriores: {len(clima_dia)}\n")
clima_dia["lluvia"] = pd.cut(clima_dia["lluvia_dia"], [-0.01, 0.1, 5, 20, np.inf],
                             labels=["seco (< 0,1 mm)", "0,1 a 5 mm", "5 a 20 mm", "más de 20 mm"])
clima_dia["temperatura"] = pd.cut(clima_dia["temp_media"], [-np.inf, 10, 15, 20, 25, np.inf],
                                  labels=["< 10 °C", "10 a 15", "15 a 20", "20 a 25", "> 25 °C"])

for col in ("lluvia", "temperatura"):
    t = clima_dia.groupby(col, observed=False)["cociente"].agg(dias="count", cociente_medio="mean")
    print(f"Días hábiles por {col} pronosticada:")
    print(t.round(3).to_string(), "\n")

## 6. Relación entre variables

Usamos la **correlación de Pearson**. Es un número entre -1 y 1 que dice qué tan bien dos variables se acomodan sobre una línea recta:
- Cerca de 1: cuando una sube, la otra también sube.
- Cerca de -1: cuando una sube, la otra baja.
- Cerca de 0: no hay relación en línea recta.

Se calcula sobre una muestra de 200.000 filas completas de train. Lo miramos de dos formas:

1. **Con `pasajeros`**: qué variables se mueven junto con lo que queremos predecir.
2. **Entre ellas**: si dos variables son casi iguales (correlación cerca de 1), dicen lo mismo. En una regresión lineal eso es un problema, porque el modelo no sabe a cuál darle el peso y los coeficientes quedan inestables (cambian mucho con cambios chicos en los datos). Ejemplo: `pasajeros_prom_4sem` es exactamente el promedio de los rezagos de 7, 14, 21 y 28 días. No tiene sentido poner los cinco juntos en una regresión lineal.

**Cuidado con la hora.** Su correlación con `pasajeros` va a dar baja, aunque la hora es de lo que más importa (ver 3.1). Pasa porque la correlación solo mide relaciones en línea recta, y los pasajeros por hora tienen forma de dos montañas (mañana y tarde). Un número solo no alcanza para describir una relación, siempre hay que mirar el gráfico.

In [0]:
NUMERICAS = [
    "pasajeros",
    "pasajeros_lag_1d", "pasajeros_lag_7d", "pasajeros_lag_14d", "pasajeros_lag_21d", "pasajeros_lag_28d",
    "pasajeros_prom_4sem", "pasajeros_mediana_4sem", "pasajeros_linea_dia_anterior",
    "temperatura_pron", "lluvia_pron_mm", "hora", "dia_semana",
]
completas = train.where(F.col("fila_completa"))
n = completas.count()
muestra = (completas.sample(fraction=min(1.0, 200_000 / max(n, 1)), seed=config.SEED)
           .select(*NUMERICAS).toPandas().astype(float))
corr = muestra.corr()

nombres = [c.replace("pasajeros_", "") for c in NUMERICAS]
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(nombres)), nombres, rotation=60, ha="right")
ax.set_yticks(range(len(nombres)), nombres)
ax.grid(False)
for i in range(len(nombres)):
    for j in range(len(nombres)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im)
ax.set_title(f"Correlación de Pearson (muestra de {len(muestra):,} filas completas de train)")
plt.tight_layout()
plt.show()

print("Correlación con pasajeros:")
print(corr["pasajeros"].drop("pasajeros").sort_values(ascending=False).round(3).to_string())

## Conclusiones

**Ojo: hay que actualizar estos números.** Los escribimos cuando train era de enero a setiembre de 2024 (136 líneas, 274 días, 894.336 filas). Ahora train es todo 2024, así que hay que volver a correr el notebook y cambiar los números de la tabla. Las decisiones no deberían cambiar, pero hay que confirmarlo.

| Pregunta | Qué encontramos | Qué decidimos |
|---|---|---|
| ¿Hay datos vacíos que no esperábamos? | No. Solo faltan rezagos (enero), clima (antes del 19/01) y `nombre_dia_especial` | Los modelos usan solo `fila_completa` (89,8 % de las filas) |
| ¿Los pasajeros están muy torcidos hacia valores grandes? | Sí. Media 251, mediana 67, asimetría 2,03. El 24,4 % de las horas tiene 0 pasajeros | Probar log(1 + pasajeros) en el modelo lineal. El error siempre se mide en pasajeros |
| ¿La forma del día cambia según el tipo de día? | Sí. Hábil: pico a las 7 y 8, baja a las 10, sube y se mantiene de 12 a 18. Sábado: una sola loma (13 h). Domingo y feriado: bajo y parejo. Turismo se parece al sábado | Agregar la combinación hora x tipo de día. Crear `tipo_dia`. Turismo se podría juntar con sábado |
| ¿Hay días bajos sin explicación (posibles paros)? | No. Los dos días hábiles bajos son feriados laborables (22/04 y 19/06) | La variable de paros no es urgente |
| ¿Pocas líneas llevan casi todos los pasajeros? | Las 10 más grandes llevan el 24 %, las 20 más grandes el 42 % y las 50 más grandes el 80 % | Mostrar el error total y también por tamaño de línea |
| ¿Las líneas grandes varían más? | Sí, casi en proporción a su tamaño (recta en el gráfico) | Otra razón para usar log(1 + pasajeros) |
| ¿Cuánto cambian los pasajeros con clases, feriados y lluvia? | Con clases (escuela y liceo) viaja un 25 % más. En feriado laborable un 36 % menos (solo 2 días). Con lluvia de 5 a 20 mm un 15 % menos que un día seco. Temperatura: no se puede saber (se mezcla con la vuelta de vacaciones). Elecciones: hay pocos días (en 2024 fueron tres) | Usar clases, feriados y lluvia del día. La temperatura con cuidado |
| ¿Qué variables dicen lo mismo? | Los rezagos semanales, el promedio y la mediana tienen correlación de 0,95 a 1,00 entre sí | En la regresión lineal usar solo `pasajeros_mediana_4sem` y `pasajeros_lag_1d` (y como mucho `lag_7d`) |
| ¿Qué tan difícil es ganarle a una regla simple? | `pasajeros_mediana_4sem` tiene correlación 0,972 con `pasajeros` | El R² no sirve para comparar modelos. Hay que compararlos contra la mediana de 4 semanas |

**Lo más importante.** La cantidad de pasajeros se repite mucho de una semana a otra. Por eso la regla "lo mismo que en las últimas 4 semanas" casi siempre acierta. Un modelo solo puede ganar en los días en que algo cambia: feriados, comienzo de clases, lluvia fuerte, Turismo. En el notebook 06 se separa el error por tipo de día para mostrarlo.

**Límites.** En train hay pocos feriados laborables y solo tres elecciones (internas, nacionales y balotaje). No se puede saber el efecto de la temperatura. Además, las filas de un mismo día se parecen mucho entre sí (son unos 340 días completos, no un millón de datos independientes). Por eso los intervalos de confianza de los errores se calculan sorteando días y no filas.

In [0]:
train.unpersist()